# Detección de enfermedades en hojas de plantas con transfer learning

Proyecto final de Machine Learning (UTEC). Clasificación de hojas sanas vs. enfermas con el dataset PlantVillage y ResNet50 preentrenada en ImageNet.

Ejecutar en Google Colab con GPU (Entorno de ejecución > Cambiar tipo de entorno > GPU).

## 1. Configuración

In [ ]:
import os, json, zipfile
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from PIL import Image
from huggingface_hub import hf_hub_download
from sklearn.model_selection import GroupShuffleSplit
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, roc_auc_score, RocCurveDisplay)

SEED = 42
tf.keras.utils.set_random_seed(SEED)

MODO = "binario"      # "binario" (sana vs. enferma) o "multiclase" (38 clases)
IMG_SIZE = 224
BATCH = 32
FRACCION = 1.0        # bajar a 0.3 si el entrenamiento tarda demasiado
EPOCAS_FASE1 = 5
EPOCAS_FASE2 = 5
DATA_DIR = "plantvillage"

print("TensorFlow", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))

## 2. Descarga del dataset

Descargamos PlantVillage desde Hugging Face (~2 GB) junto con la partición oficial 80/20, que agrupa las imágenes por hoja para evitar fuga de información entre train y test, y el mapa que identifica a qué hoja física pertenece cada foto.

In [ ]:
REPO = "mohanty/PlantVillage"
archivos = {f: hf_hub_download(REPO, f, repo_type="dataset", local_dir=DATA_DIR)
            for f in ["splits/color_train.txt", "splits/color_test.txt", "leaf_grouping/leaf-map.json"]}

if not os.path.isdir(os.path.join(DATA_DIR, "raw", "color")):
    zip_path = hf_hub_download(REPO, "data.zip", repo_type="dataset", local_dir=DATA_DIR)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(DATA_DIR, members=[m for m in z.namelist() if m.startswith("raw/color/")])
    os.remove(zip_path)

with open(archivos["leaf_grouping/leaf-map.json"]) as f:
    LEAF_MAP = json.load(f)
print(len(os.listdir(os.path.join(DATA_DIR, "raw", "color"))), "carpetas de clase")

In [ ]:
def id_hoja(clase, archivo):
    # Misma lógica que el script oficial del dataset para identificar la hoja física
    ident = archivo.replace("_final_masked", "")
    if "___" in ident:
        ident = ident.split("___")[-1]
    ident = ident.split("copy")[0]
    for ext in (".jpg", ".JPG", ".png", ".PNG"):
        ident = ident.replace(ext, "")
    ident = ident.strip()
    sugerencias = LEAF_MAP.get(ident.lower().strip())
    if not sugerencias:
        return "fallback_" + ident
    if len(sugerencias) == 1:
        return sugerencias[0]
    return next((s for s in sugerencias if clase in s), "fallback_" + ident)

def leer_split(nombre):
    with open(archivos[f"splits/color_{nombre}.txt"]) as f:
        rutas = [l.strip() for l in f if l.strip()]
    clases = [r.split("/")[2] for r in rutas]
    hojas = [id_hoja(c, r.split("/")[3]) for c, r in zip(clases, rutas)]
    return np.array([os.path.join(DATA_DIR, r) for r in rutas]), np.array(clases), np.array(hojas)

rutas_tr, clases_tr, hojas_tr = leer_split("train")
rutas_te, clases_te, hojas_te = leer_split("test")

CLASES_38 = sorted(set(clases_tr))
if MODO == "binario":
    CLASES = ["sana", "enferma"]
    objetivo = lambda cl: np.array([0 if "healthy" in c.lower() else 1 for c in cl])
else:
    CLASES = CLASES_38
    objetivo = lambda cl: np.array([CLASES_38.index(c) for c in cl])
N_CLASES = len(CLASES)

print(len(rutas_tr), "imágenes de train |", len(rutas_te), "de test |", N_CLASES, "clases")

## 3. Submuestreo opcional y partición de validación por hoja

Separamos un 15% del train como validación usando el identificador de hoja, para que una misma hoja no aparezca en train y validación.

In [ ]:
rng = np.random.default_rng(SEED)
if FRACCION < 1.0:
    i_tr = rng.choice(len(rutas_tr), int(len(rutas_tr) * FRACCION), replace=False)
    i_te = rng.choice(len(rutas_te), int(len(rutas_te) * FRACCION), replace=False)
    rutas_tr, clases_tr, hojas_tr = rutas_tr[i_tr], clases_tr[i_tr], hojas_tr[i_tr]
    rutas_te, clases_te = rutas_te[i_te], clases_te[i_te]

gss = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=SEED)
i_fit, i_val = next(gss.split(rutas_tr, groups=hojas_tr))

X_train, y_train = rutas_tr[i_fit], objetivo(clases_tr[i_fit])
X_val, y_val = rutas_tr[i_val], objetivo(clases_tr[i_val])
X_test, y_test = rutas_te, objetivo(clases_te)

assert not set(hojas_tr[i_fit]) & set(hojas_tr[i_val]), "Hay hojas repetidas entre train y validación"
print("train:", len(X_train), "| val:", len(X_val), "| test:", len(X_test))

## 4. EDA: distribución de clases y ejemplos

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4), gridspec_kw={"width_ratios": [1, 3]})
v, n = np.unique(y_train, return_counts=True)
ax[0].bar([CLASES[i] if MODO == "binario" else str(i) for i in v], n)
ax[0].set_title("Distribución de la variable objetivo (train)")
cl, n38 = np.unique(clases_tr, return_counts=True)
orden = np.argsort(n38)
ax[1].barh(cl[orden], n38[orden]); ax[1].tick_params(axis="y", labelsize=6)
ax[1].set_title("Imágenes por clase original (38 clases)")
plt.tight_layout(); plt.savefig("distribucion_clases.png", dpi=150); plt.show()

fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for a, i in zip(axes.ravel(), rng.choice(len(rutas_tr), 10, replace=False)):
    a.imshow(Image.open(rutas_tr[i])); a.set_title(clases_tr[i][:28], fontsize=7); a.axis("off")
plt.tight_layout(); plt.savefig("ejemplos.png", dpi=150); plt.show()

print("Tamaño de imagen original:", Image.open(rutas_tr[0]).size)

## 5. Pipeline: reescalado y data augmentation

La normalización con las estadísticas de ImageNet se aplica dentro del modelo (`preprocess_input`), igual que en el preentrenamiento de ResNet50.

In [ ]:
def cargar(ruta, etiqueta):
    img = tf.io.decode_jpeg(tf.io.read_file(ruta), channels=3)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    return img, etiqueta

def a_dataset(X, y, entrenamiento):
    d = tf.data.Dataset.from_tensor_slices((X, y))
    if entrenamiento:
        d = d.shuffle(len(X), seed=SEED, reshuffle_each_iteration=True)
    return d.map(cargar, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH).prefetch(tf.data.AUTOTUNE)

train_ds = a_dataset(X_train, y_train, True)
val_ds = a_dataset(X_val, y_val, False)
test_ds = a_dataset(X_test, y_test, False)

augment = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal_and_vertical"),
    tf.keras.layers.RandomRotation(0.15),
    tf.keras.layers.RandomZoom(0.15),
    tf.keras.layers.RandomBrightness(0.15, value_range=(0, 255)),
    tf.keras.layers.RandomContrast(0.15),
], name="augmentation")

## 6. Modelo: ResNet50 preentrenada + cabeza de clasificación

In [ ]:
base = tf.keras.applications.ResNet50(include_top=False, weights="imagenet",
                                      input_shape=(IMG_SIZE, IMG_SIZE, 3))
base.trainable = False

entrada = tf.keras.Input((IMG_SIZE, IMG_SIZE, 3))
x = augment(entrada)
x = tf.keras.layers.Lambda(tf.keras.applications.resnet50.preprocess_input, name="normalizacion_imagenet")(x)
x = base(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.3)(x)
reg = tf.keras.regularizers.l2(1e-4)
if MODO == "binario":
    salida = tf.keras.layers.Dense(1, activation="sigmoid", kernel_regularizer=reg)(x)
    perdida = "binary_crossentropy"
    metricas = lambda: ["accuracy", tf.keras.metrics.AUC(name="auc"),
                        tf.keras.metrics.Recall(name="recall"), tf.keras.metrics.Precision(name="precision")]
else:
    salida = tf.keras.layers.Dense(N_CLASES, activation="softmax", kernel_regularizer=reg)(x)
    perdida = "sparse_categorical_crossentropy"
    metricas = lambda: ["accuracy"]
modelo = tf.keras.Model(entrada, salida)
modelo.summary(show_trainable=True)

## 7. Entrenamiento en dos fases con pesos de clase y early stopping

In [ ]:
pesos = compute_class_weight("balanced", classes=np.unique(y_train), y=y_train)
class_weight = {int(k): float(w) for k, w in zip(np.unique(y_train), pesos)}
print("Pesos de clase:", class_weight)

def callbacks():
    return [tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
            tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=2)]

# Fase 1: base congelada, solo se entrena la cabeza
modelo.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss=perdida, metrics=metricas())
h1 = modelo.fit(train_ds, validation_data=val_ds, epochs=EPOCAS_FASE1,
                class_weight=class_weight, callbacks=callbacks())

In [ ]:
# Fase 2: fine-tuning del último bloque convolucional (conv5) con learning rate bajo.
# Las capas de BatchNormalization se mantienen congeladas para no romper sus estadísticas.
base.trainable = True
for capa in base.layers:
    capa.trainable = capa.name.startswith("conv5") and not isinstance(capa, tf.keras.layers.BatchNormalization)
print("Capas entrenables en la base:", sum(c.trainable for c in base.layers))

modelo.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss=perdida, metrics=metricas())
h2 = modelo.fit(train_ds, validation_data=val_ds, epochs=EPOCAS_FASE2,
                class_weight=class_weight, callbacks=callbacks())
modelo.save("resnet50_plantvillage.keras")

In [ ]:
def unir(k): return h1.history[k] + h2.history[k]
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for i, k in enumerate(["loss", "accuracy"]):
    ax[i].plot(unir(k), marker="o", label="train"); ax[i].plot(unir("val_" + k), marker="o", label="validación")
    ax[i].axvline(len(h1.history["loss"]) - 0.5, ls="--", c="gray", label="inicio fine-tuning")
    ax[i].set_title(k); ax[i].set_xlabel("época"); ax[i].legend()
plt.tight_layout(); plt.savefig("curvas_entrenamiento.png", dpi=150); plt.show()

## 8. Evaluación en test

In [ ]:
prob = modelo.predict(test_ds)
if MODO == "binario":
    prob = prob.ravel()
    y_pred = (prob >= 0.5).astype(int)
    auc = roc_auc_score(y_test, prob)
else:
    y_pred = prob.argmax(1)
    presentes = np.unique(y_test)
    auc = roc_auc_score(y_test, prob[:, presentes] / prob[:, presentes].sum(1, keepdims=True),
                        multi_class="ovr", average="macro", labels=presentes)

print(classification_report(y_test, y_pred, labels=range(N_CLASES), target_names=CLASES,
                            digits=4, zero_division=0))
print("AUC:", round(auc, 4))

fig, ax = plt.subplots(figsize=(5, 5) if MODO == "binario" else (14, 14))
ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred, labels=range(N_CLASES)),
                       display_labels=CLASES if MODO == "binario" else None).plot(ax=ax, cmap="Greens", colorbar=False)
ax.set_title("Matriz de confusión (test)"); plt.savefig("matriz_confusion.png", dpi=150); plt.show()

if MODO == "binario":
    RocCurveDisplay.from_predictions(y_test, prob)
    plt.title(f"Curva ROC (AUC = {auc:.4f})"); plt.savefig("curva_roc.png", dpi=150); plt.show()

## 9. Análisis de errores

Ejemplos mal clasificados para la sección de Discusión.

In [ ]:
errores = np.where(y_pred != y_test)[0]
print(len(errores), "errores de", len(y_test))
if len(errores):
    fig, axes = plt.subplots(2, 5, figsize=(14, 6))
    for a in axes.ravel(): a.axis("off")
    for a, i in zip(axes.ravel(), errores[:10]):
        a.imshow(Image.open(X_test[i]))
        a.set_title(f"real: {CLASES[y_test[i]][:18]}\npred: {CLASES[y_pred[i]][:18]}\n{clases_te[i][:24]}", fontsize=7)
    plt.tight_layout(); plt.savefig("errores.png", dpi=150); plt.show()